# 07 — Análisis de Sentimiento YouTube

Inferencia de sentimiento sobre el corpus limpio de comentarios de YouTube.
Modelo: `cardiffnlp/twitter-xlm-roberta-base-sentiment` — mismo que Twitter,
imprescindible para comparabilidad cross-platform (H3).

Input:  `data/processed/youtube_comments_clean.parquet` — 62.989 comentarios
Output: `data/processed/youtube_sentiment.parquet` — con columnas sentiment_label y sentiment_score

**Nota:** las celdas de inferencia (3-5) no tienen output guardado porque el proceso
se ejecutó en una sesión anterior. Los resultados están correctamente guardados
en el parquet de output y se verifican en la celda final.

In [1]:
import psutil
ram = psutil.virtual_memory()
print(f"RAM disponible: {ram.available / 1024**3:.1f} GB")
print(f"RAM usada: {ram.used / 1024**3:.1f} GB")

RAM disponible: 3.7 GB
RAM usada: 8.0 GB


In [2]:
import pandas as pd
import torch
import gc
import os
from transformers import pipeline

print(f"RAM disponible antes de cargar modelo: comprueba que tienes >3GB libres")
print(f"CUDA: {torch.cuda.is_available()} — corriendo en CPU")

df = pd.read_parquet("../data/processed/youtube_comments_clean.parquet")
print(f"Comentarios a procesar: {len(df):,}")

RAM disponible antes de cargar modelo: comprueba que tienes >3GB libres
CUDA: False — corriendo en CPU
Comentarios a procesar: 62,989


## Carga del modelo

Se usa `device=-1` para forzar CPU ya que no hay GPU disponible.
`batch_size=8` es conservador para no saturar los 11.7 GB de RAM del sistema.
`truncation=True` con `max_length=512` gestiona comentarios largos (máx. 1.888 palabras
en el corpus) truncando de forma controlada.

El warning de `position_ids` es esperado y puede ignorarse: es un artefacto de
compatibilidad entre versiones de la librería, no afecta a los resultados.

In [3]:
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

# Mismo modelo que Twitter — imprescindible para comparabilidad cross-platform (H3)
sentiment_pipeline = pipeline(
    task="text-classification",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=-1,          # -1 = CPU obligatorio
    max_length=512,
    truncation=True,
    batch_size=8        # conservador por RAM limitada
)

print("Modelo cargado correctamente")

# Liberar memoria no usada inmediatamente
gc.collect()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo cargado correctamente


132

## Sistema de inferencia con checkpoint

La inferencia sobre 62.989 comentarios en CPU tarda aproximadamente 4-6 horas.
Para evitar pérdida de trabajo ante caídas del kernel, se implementa un sistema
de checkpoint que guarda el progreso cada 5.000 comentarios en:
`data/processed/yt_sentiment_checkpoint.parquet`

Si la ejecución se interrumpe, al relanzar la función detecta el checkpoint
automáticamente y continúa desde el último punto guardado.

In [4]:
CHECKPOINT_PATH = "../data/processed/yt_sentiment_checkpoint.parquet"
BATCH_SIZE = 8
CHECKPOINT_EVERY = 5000  # Guardar progreso cada 5.000 comentarios

def run_sentiment_with_checkpoint(df, pipeline, checkpoint_path):
    
    # Comprobar si existe checkpoint previo
    if os.path.exists(checkpoint_path):
        checkpoint = pd.read_parquet(checkpoint_path)
        done_ids = set(checkpoint["comment_id"].tolist())
        df_pending = df[~df["comment_id"].isin(done_ids)].copy()
        results = checkpoint.to_dict("records")
        print(f"Checkpoint encontrado: {len(done_ids):,} ya procesados")
        print(f"Pendientes: {len(df_pending):,}")
    else:
        df_pending = df.copy()
        results = []
        print(f"Sin checkpoint previo. Procesando {len(df_pending):,} comentarios")
    
    texts = df_pending["text_clean"].tolist()
    ids = df_pending["comment_id"].tolist()
    total = len(texts)
    
    for i in range(0, total, BATCH_SIZE):
        batch_texts = texts[i:i+BATCH_SIZE]
        batch_ids = ids[i:i+BATCH_SIZE]
        
        try:
            preds = pipeline(batch_texts)
        except Exception as e:
            print(f"Error en batch {i}-{i+BATCH_SIZE}: {e}")
            # Rellenar con None para no perder el índice
            preds = [{"label": None, "score": None}] * len(batch_texts)
        
        for cid, pred in zip(batch_ids, preds):
            results.append({
                "comment_id": cid,
                "sentiment_label": pred["label"],
                "sentiment_score": pred["score"]
            })
        
        # Progreso y checkpoint
        processed = len(results)
        if processed % CHECKPOINT_EVERY < BATCH_SIZE:
            pd.DataFrame(results).to_parquet(checkpoint_path, index=False)
            pct = processed / (total + len(done_ids if 'done_ids' in dir() else [])) * 100
            print(f"  Checkpoint guardado: {processed:,} procesados ({pct:.1f}%)")
    
    return pd.DataFrame(results)

## Ejecución de la inferencia

Tiempo total de ejecución en CPU (Intel i5, sin GPU): ~5 horas.
La celda usa `%%time` para registrar el tiempo exacto.

In [ ]:
%%time
# Esta celda puede tardar 4-6 horas en CPU
# NO cierres el kernel mientras corre

print("Iniciando inferencia de sentimiento...")
print("Puedes monitorizar el progreso por los mensajes de checkpoint")
print("-" * 50)

df_sentiment = run_sentiment_with_checkpoint(
    df=df,
    pipeline=sentiment_pipeline,
    checkpoint_path=CHECKPOINT_PATH
)

print(f"\nInferencia completada: {len(df_sentiment):,} comentarios procesados")
print(df_sentiment["sentiment_label"].value_counts())

Iniciando inferencia de sentimiento...
Puedes monitorizar el progreso por los mensajes de checkpoint
--------------------------------------------------
Sin checkpoint previo. Procesando 62,989 comentarios


## Unión de resultados y guardado

Se hace merge por `comment_id` para añadir `sentiment_label` y `sentiment_score`
al dataframe original. Se verifica que no haya nulos antes de guardar.
El checkpoint se elimina automáticamente si la inferencia fue completa y limpia.

In [ ]:
# Unir sentimiento con el dataframe original
df_final = df.merge(df_sentiment, on="comment_id", how="left")

# Verificar que no hay nulos inesperados
nulos_sentiment = df_final["sentiment_label"].isnull().sum()
print(f"Comentarios sin sentimiento asignado: {nulos_sentiment:,}")
if nulos_sentiment > 0:
    print("AVISO: hay comentarios sin clasificar. Revisar errores en la inferencia.")

# Distribución de sentimiento
print("\nDistribución de sentimiento:")
print(df_final["sentiment_label"].value_counts())
print(df_final["sentiment_label"].value_counts(normalize=True).apply(lambda x: f"{x*100:.1f}%"))

# Guardar resultado final
df_final.to_parquet(
    "../data/processed/youtube_sentiment.parquet",
    index=False
)
print("\nGuardado en data/processed/youtube_sentiment.parquet")

# Eliminar checkpoint si todo fue bien
if nulos_sentiment == 0:
    os.remove(CHECKPOINT_PATH)
    print("Checkpoint eliminado — inferencia completa y limpia")

In [ ]:
# Esto es el primer vistazo a los resultados — la señal de que H2 tiene sentido
pivot = df_final.groupby(["topic", "sentiment_label"]).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

print("Distribución de sentimiento por topic (%):")
print(pivot_pct.round(1))

## Verificación de resultados — resumen ejecutivo

Carga directa desde el parquet guardado para verificar integridad del output.

**Hallazgos principales:**
- Sentimiento global: 52.9% negativo, 26.0% neutral, 21.1% positivo
- **H1 confirmada:** tendencia negativa creciente y monótona 2020→2026.
  El negativismo pasa del 35.3% en 2021 (mínimo) al 60.7% en 2026.
  Punto de inflexión visible: salto de 7.6pp entre 2022 (39.7%) y 2023 (47.3%).
- **H2 confirmada parcialmente:** employment es el sector más negativo (59.6%)
  y education el menos negativo (47.0%), diferencia de 12.6pp.
- **"Both" (59.9% negativo):** los comentarios que cruzan educación y empleo
  son los más pesimistas del corpus, coherente con el discurso de desplazamiento laboral.

**Limitación a documentar:** 2026 solo cubre enero-marzo, lo que puede introducir
sesgo si hubo algún evento concreto en ese período que inflara el negativismo.

In [7]:
import pandas as pd

df = pd.read_parquet("../data/processed/youtube_sentiment.parquet")

print(f"Total comentarios: {len(df):,}")
print(f"\nDistribución de sentimiento:")
print(df["sentiment_label"].value_counts())
print(df["sentiment_label"].value_counts(normalize=True).apply(lambda x: f"{x*100:.1f}%"))

print(f"\nSentimiento por topic:")
pivot = df.groupby(["topic", "sentiment_label"]).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
print(pivot_pct.round(1))

print(f"\nSentimiento por año:")
pivot_anual = df.groupby(["comment_year", "sentiment_label"]).size().unstack(fill_value=0)
pivot_anual_pct = pivot_anual.div(pivot_anual.sum(axis=1), axis=0) * 100
print(pivot_anual_pct.round(1))

Total comentarios: 62,989

Distribución de sentimiento:
sentiment_label
negative    33341
neutral     16368
positive    13280
Name: count, dtype: int64
sentiment_label
negative    52.9%
neutral     26.0%
positive    21.1%
Name: proportion, dtype: object

Sentimiento por topic:
sentiment_label  negative  neutral  positive
topic                                       
both                 59.9     26.4      13.7
education            47.0     28.0      25.0
employment           59.6     22.6      17.8
general              51.8     26.3      21.9

Sentimiento por año:
sentiment_label  negative  neutral  positive
comment_year                                
2020                 42.0     33.9      24.1
2021                 35.3     39.4      25.3
2022                 39.7     34.1      26.2
2023                 47.3     28.4      24.2
2024                 52.5     26.2      21.3
2025                 55.9     23.4      20.6
2026                 60.7     22.4      16.8
